Stage 1: Extract Landmark bằng mediapipe, tạo data type là json để lưu tạm thời dữ liệu trên RAM
Stage 2: Tính toán thành face feature, CSV trên RAM
Stage 3: Nén face feature, output CSV

Dữ liệu cuối cùng là tabular bên trong CSV nên ta dùng Decision Tree, Random Forest và XGboost

In [1]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg


In [1]:

img = mpimg.imread(r"C:\Users\KIET PC\Desktop\Focus_Guard\Screenshot 2026-08-08 115540.png")
plt.figure(figsize=(12, 8))
plt.imshow(img)
plt.axis("off")
plt.show()

NameError: name 'mpimg' is not defined

In [ ]:
import cv2                  #Input ảnh vào
import mediapipe as mp    #extract landmark

# 1. Khởi tạo MediaPipe Face Landmarker
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=False,        # False cho video, True cho ảnh tĩnh
    max_num_faces=1,                # Số khuôn mặt tối đa muốn detect
    min_detection_confidence=0.5,   # Ngưỡng tin cậy để detect
    min_tracking_confidence=0.5     # Ngưỡng tin cậy để tracking
)

# 2. Helper để vẽ
mp_drawing = mp.solutions.drawing_utils
drawing_spec = mp_drawing.DrawingSpec(thickness=1, circle_radius=1)

# 3. Mở webcam
cap = cv2.VideoCapture(0)

while cap.isOpened():
    success, image = cap.read()
    if not success:
        break
    
    # Chuyển BGR sang RGB (MediaPipe yêu cầu RGB)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image_rgb.flags.writeable = False
    
    # 4. Xử lý: extract landmarks
    results = face_mesh.process(image_rgb)
    
    # Chuyển lại BGR để hiển thị
    image_rgb.flags.writeable = True
    image = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
    
    # 5. Vẽ landmarks lên ảnh
    if results.multi_face_landmarks:
        for face_landmarks in results.multi_face_landmarks:
            # Vẽ toàn bộ 468 điểm (dạng lưới)
            mp_drawing.draw_landmarks(
                image=image,
                landmark_list=face_landmarks,
                connections=mp_face_mesh.FACEMESH_CONTOURS,  # Các kết nối (đường viền)
                landmark_drawing_spec=drawing_spec,
                connection_drawing_spec=drawing_spec
            )
            
            # --- LẤY TỌA ĐỘ CỤ THỂ CỦA CÁC LANDMARK ---
            h, w, _ = image.shape
            for idx, landmark in enumerate(face_landmarks.landmark):
                x = int(landmark.x * w)
                y = int(landmark.y * h)
                z = landmark.z  # Độ sâu tương đối
                
                # Ví dụ: In ra tọa độ của 5 điểm đầu tiên
                if idx < 5:
                    print(f"Landmark {idx}: x={x}, y={y}, z={z:.3f}")
    
    # 6. Hiển thị
    cv2.imshow('MediaPipe Face Landmarks', image)
    
    if cv2.waitKey(5) & 0xFF == ord('q'):
        break

# 7. Giải phóng tài nguyên
face_mesh.close()
cap.release()
cv2.destroyAllWindows()